In [8]:
import duckdb
import chromadb
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Path resolution
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CORPUS_PATH = PROJECT_ROOT / "data" / "processed" / "corpus.txt"

# Load corpus
with open(CORPUS_PATH) as f:
    all_sentences = [line.strip() for line in f.readlines()]

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Corpus: {len(all_sentences)} sentences")
print(f"Model: {model.get_embedding_dimension()}d embeddings")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Corpus: 28315 sentences
Model: 384d embeddings


In [9]:
# Initialize ChromaDB with persistent storage
chroma_client = chromadb.PersistentClient(
    path=str(PROJECT_ROOT / "chroma_db")
)

# Delete collection if it exists (clean slate)
try:
    chroma_client.delete_collection("f1_corpus")
    print("Deleted existing collection")
except:
    pass

collection = chroma_client.create_collection(
    name="f1_corpus",
    metadata={"hnsw:space": "cosine"}
)

print(f"Collection created: {collection.name}")
print("Embedding and indexing all sentences (this takes 2-3 minutes)...")

# Index in batches of 500
batch_size = 500
for i in range(0, len(all_sentences), batch_size):
    batch = all_sentences[i:i+batch_size]
    embeddings = model.encode(batch).tolist()
    ids = [str(j) for j in range(i, i+len(batch))]
    
    collection.add(
        documents=batch,
        embeddings=embeddings,
        ids=ids
    )
    print(f"  Indexed {min(i+batch_size, len(all_sentences))}/{len(all_sentences)}")

print(f"\nDone. Total indexed: {collection.count()}")

Deleted existing collection
Collection created: f1_corpus
Embedding and indexing all sentences (this takes 2-3 minutes)...
  Indexed 500/28315
  Indexed 1000/28315
  Indexed 1500/28315
  Indexed 2000/28315
  Indexed 2500/28315
  Indexed 3000/28315
  Indexed 3500/28315
  Indexed 4000/28315
  Indexed 4500/28315
  Indexed 5000/28315
  Indexed 5500/28315
  Indexed 6000/28315
  Indexed 6500/28315
  Indexed 7000/28315
  Indexed 7500/28315
  Indexed 8000/28315
  Indexed 8500/28315
  Indexed 9000/28315
  Indexed 9500/28315
  Indexed 10000/28315
  Indexed 10500/28315
  Indexed 11000/28315
  Indexed 11500/28315
  Indexed 12000/28315
  Indexed 12500/28315
  Indexed 13000/28315
  Indexed 13500/28315
  Indexed 14000/28315
  Indexed 14500/28315
  Indexed 15000/28315
  Indexed 15500/28315
  Indexed 16000/28315
  Indexed 16500/28315
  Indexed 17000/28315
  Indexed 17500/28315
  Indexed 18000/28315
  Indexed 18500/28315
  Indexed 19000/28315
  Indexed 19500/28315
  Indexed 20000/28315
  Indexed 20500/2

In [10]:
def retrieve(query: str, top_k: int = 5) -> list[dict]:
    """
    Embed a query and find the top_k most similar sentences in ChromaDB.
    This is the R in RAG.
    """
    query_embedding = model.encode([query]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    
    retrieved = []
    for doc, distance in zip(results["documents"][0], results["distances"][0]):
        retrieved.append({
            "text": doc,
            "score": 1 - distance  # ChromaDB cosine returns distance, convert to similarity
        })
    
    return retrieved

# Test it
results = retrieve("Who won the most races for Mercedes?")
print("Query: 'Who won the most races for Mercedes?'\n")
for r in results:
    print(f"[{r['score']:.4f}] {r['text']}")

Query: 'Who won the most races for Mercedes?'

[0.6750] At the end of the 2026 season, antonelli led the drivers championship with 72.0 points and 2 wins, driving for Mercedes.
[0.6588] At the end of the 1954 season, kling was 5th in the drivers championship with 12.0 points and 0 wins, driving for Mercedes.
[0.6569] At the end of the 1955 season, kling was 11th in the drivers championship with 5.0 points and 0 wins, driving for Mercedes.
[0.6563] At the end of the 1955 season, fangio led the drivers championship with 40.0 points and 4 wins, driving for Mercedes.
[0.6424] At the end of the 2025 season, antonelli was 7th in the drivers championship with 150.0 points and 0 wins, driving for Mercedes.


In [11]:
import ollama

OLLAMA_HOST = "http://172.22.80.1:11434"
client = ollama.Client(host=OLLAMA_HOST)

def generate(query: str, context: list[dict]) -> str:
    """
    Given a query and retrieved context, ask Ollama to generate an answer.
    This is the G in RAG.
    """
    context_text = "\n".join([f"- {r['text']}" for r in context])
    
    prompt = f"""You are an F1 expert assistant. Answer the question using ONLY the context below.
Synthesize the context into a direct, confident answer.
If asked about performance across multiple races, summarize the pattern you see.
If the context truly doesn't contain the answer, say so in one sentence.

Context:
{context_text}


Question: {query}

Answer:"""

    response = client.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response["message"]["content"]

# Test the generator alone first
test_context = [{"text": "In the 2020 Bahrain Grand Prix, Lewis Hamilton finished 1st (winner) starting from position 1, scoring 25 points for Mercedes."}]
print(generate("Who won the 2020 Bahrain Grand Prix?", test_context))

Lewis Hamilton.


In [12]:
def ask(query: str, top_k: int = 5, verbose: bool = True) -> str:
    """
    Full RAG pipeline:
    1. Retrieve relevant sentences from ChromaDB
    2. Generate answer using Ollama with those sentences as context
    """
    # Step 1: Retrieve
    context = retrieve(query, top_k=top_k)
    
    if verbose:
        print(f"Query: {query}")
        print(f"\nRetrieved context:")
        for r in context:
            print(f"  [{r['score']:.4f}] {r['text']}")
        print()
    
    # Step 2: Generate
    answer = generate(query, context)
    
    if verbose:
        print(f"Answer:\n{answer}")
    
    return answer

In [13]:
questions = [
    "Who won the 2021 Abu Dhabi Grand Prix?",
    "Which constructor dominated the 2014 season?",
    "How did Max Verstappen perform at Monaco?",
    "Who had the most wins for Ferrari in the 2000s?",
]

for q in questions:
    print("="*60)
    ask(q)
    print()

Query: Who won the 2021 Abu Dhabi Grand Prix?

Retrieved context:
  [0.6480] In the 2025 Abu Dhabi Grand Prix, Oscar Piastri finished 2nd starting from position 3, scoring 18.0 points for McLaren.
  [0.6423] In the 2024 Abu Dhabi Grand Prix, Oscar Piastri finished 10th starting from position 2, scoring 1.0 points for McLaren.
  [0.6368] In the 2025 Bahrain Grand Prix, Oscar Piastri finished 1st (winner) starting from position 1, scoring 25.0 points for McLaren and set the fastest lap.
  [0.6333] In the 2024 Abu Dhabi Grand Prix, Sergio Pérez finished 20th starting from position 10, scoring 0.0 points for Red Bull.
  [0.6309] In the 2022 Abu Dhabi Grand Prix, Fernando Alonso finished 20th starting from position 10, scoring 0.0 points for Alpine F1 Team.

Answer:
The context only provides information about the 2024 and 2025 Abu Dhabi Grand Prix, as well as the 2025 Bahrain Grand Prix. It does not mention the 2021 Abu Dhabi Grand Prix or any other race. Therefore, I cannot provide an answ

In [14]:
# Ask something the corpus genuinely can't answer
hard_questions = [
    "What was the weather like at the 2019 German Grand Prix?",
    "How many pit stops did Hamilton make at Silverstone 2023?",
    "What is Verstappen's favourite circuit?",
]

print("Testing questions the corpus can't fully answer:\n")
for q in hard_questions:
    print("="*60)
    ask(q)
    print()

Testing questions the corpus can't fully answer:

Query: What was the weather like at the 2019 German Grand Prix?

Retrieved context:
  [0.4414] In the 2019 German Grand Prix, Max Verstappen finished 1st (winner) starting from position 2, scoring 26.0 points for Red Bull and set the fastest lap.
  [0.4328] In the 1996 French Grand Prix, Andrea Montermini finished 21th starting from position 21, scoring 0.0 points for Forti.
  [0.4275] In the 2000 German Grand Prix, Jean Alesi finished 19th starting from position 20, scoring 0.0 points for Prost.
  [0.4201] In the 1996 Brazilian Grand Prix, Andrea Montermini finished 19th starting from position 20, scoring 0.0 points for Forti.
  [0.4181] In the 1974 French Grand Prix, Jochen Mass finished 19th starting from position 18, scoring 0.0 points for Surtees.

Answer:
Unfortunately, there is not enough information in the provided context to determine what the weather was like at the 2019 German Grand Prix. The context only mentions the results

In [15]:
## What We Just Built

The full RAG pipeline in ~60 lines of code:

- **Retrieve**: embed query → cosine search ChromaDB → top-5 sentences
- **Generate**: stuff context into prompt → Ollama generates grounded answer

### Failure modes observed:
1. **Retrieval failure** — wrong sentences returned, answer is hallucinated
2. **Generation failure** — right sentences returned, LLM reasons incorrectly
3. **Coverage gap** — question needs data not in corpus (weather, pit stops)

These are the three failure modes of every RAG system.
Notebooks 5 and 6 address failure modes 1 and 3.

SyntaxError: invalid character '→' (U+2192) (1928152913.py, line 5)